# 00 — Generate Projections via nflreadpy

Automated 17-game Full-PPR projection generator built on top of
**nflreadpy** (nflverse historical stats) and the **Sleeper API**
player catalog for rookie imputation.

**Workflow**
1. Setup & dependencies
2. Ingest historical NFL player stats (most recent regular season)
3. 17-game Full-PPR pace calculation
4. Rookie & missing player imputation via Sleeper
5. Export & downstream alignment
6. End-to-end validation

## Cell 1 — Setup & Dependencies

In [1]:
import os
import json
import numpy as np
import pandas as pd
import nflreadpy as nfl

# -- Project paths ---------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR     = os.path.join(PROJECT_ROOT, "data")
NOTEBOOK_DIR = os.path.join(PROJECT_ROOT, "notebooks")

SLEEPER_CACHE    = os.path.join(DATA_DIR, "sleeper_players_raw.json")
PROJECTIONS_CSV  = os.path.join(DATA_DIR, "projections.csv")
TEMPLATE_CSV     = os.path.join(DATA_DIR, "projections_template.csv")

SKILL_POSITIONS = ["QB", "RB", "WR", "TE"]

os.makedirs(DATA_DIR, exist_ok=True)

print(f"PROJECT_ROOT -> {PROJECT_ROOT}")
print(f"DATA_DIR     -> {DATA_DIR}")
print(f"OUTPUT CSV   -> {PROJECTIONS_CSV}")

PROJECT_ROOT -> /home/hadev/Projects/Lab/fantasy-footbal-analytics
DATA_DIR     -> /home/hadev/Projects/Lab/fantasy-footbal-analytics/data
OUTPUT CSV   -> /home/hadev/Projects/Lab/fantasy-footbal-analytics/data/projections.csv


## Cell 2 — Ingest Historical NFL Player Stats

Pull the most recent completed regular season from **nflreadpy**
and aggregate offensive stats by player.  We request `summary_level="reg"`
to get season totals directly (one row per player per season).

In [2]:
# -- Determine most recent available season --------------------------
CURRENT_YEAR = 2024

print(f"Loading regular-season stats for {CURRENT_YEAR} ...")
raw_stats = nfl.load_player_stats(CURRENT_YEAR, summary_level="reg")

# Convert Polars -> Pandas (rows avoids pyarrow dependency)
raw_records = raw_stats.rows(named=True)
raw_stats = pd.DataFrame(raw_records)

print(f"Raw rows (all positions): {len(raw_stats)}")
print(f"Columns: {len(raw_stats.columns)}")

# -- Filter to skill positions ---------------------------------------
skill = raw_stats[raw_stats["position"].isin(SKILL_POSITIONS)].copy()
print(f"\nSkill-position rows: {len(skill)}")
print(skill["position"].value_counts().sort_index().to_string())

Loading regular-season stats for 2024 ...


Raw rows (all positions): 1997
Columns: 148

Skill-position rows: 589
position
QB     78
RB    148
TE    127
WR    236


## Cell 3 — 17-Game Full-PPR Pace Calculation

Full-PPR scoring rules applied:

| Category | Formula |
|----------|---------|
| Passing yards | 0.04 pts/yd |
| Passing TD | +4.0 pts |
| Interception | −2.0 pts |
| Rushing yards | 0.1 pts/yd |
| Rushing TD | +6.0 pts |
| Receiving yards | 0.1 pts/yd |
| Receiving TD | +6.0 pts |
| Reception | +1.0 pts (PPR) |
| Fumble lost | −2.0 pts |

The per-game pace is scaled to 17 games.  Players with < 6 games
have their divisor floored at 6 to avoid small-sample inflation.

In [3]:
def calculate_ppr_points(df: pd.DataFrame) -> pd.Series:
    """Calculate Full-PPR fantasy points from nflreadpy stat columns."""
    pts = pd.Series(0.0, index=df.index)

    # Passing: 0.04 yds, 4 TD, -2 INT
    pts += df["passing_yards"].fillna(0) * 0.04
    pts += df["passing_tds"].fillna(0) * 4.0
    pts += df["passing_interceptions"].fillna(0) * (-2.0)

    # Rushing: 0.1 yds, 6 TD
    pts += df["rushing_yards"].fillna(0) * 0.1
    pts += df["rushing_tds"].fillna(0) * 6.0

    # Receiving: 0.1 yds, 6 TD, 1.0 PPR per reception
    pts += df["receiving_yards"].fillna(0) * 0.1
    pts += df["receiving_tds"].fillna(0) * 6.0
    pts += df["receptions"].fillna(0) * 1.0

    # Fumbles lost (sack + rushing + receiving combined)
    fumbles_lost = (
        df["sack_fumbles_lost"].fillna(0)
        + df["rushing_fumbles_lost"].fillna(0)
        + df["receiving_fumbles_lost"].fillna(0)
    )
    pts += fumbles_lost * (-2.0)

    return pts


# -- Apply scoring and project 17 games -----------------------------
skill = skill[skill["games"] >= 3].copy()
skill["total_ppr"] = calculate_ppr_points(skill)

# Floor the games divisor at 6 to avoid small-sample outliers
effective_games = skill["games"].clip(lower=6)
skill["proj_points"] = (skill["total_ppr"] / effective_games) * 17.0

# -- Build clean projection frame -----------------------------------
projections = skill[["player_id", "player_name", "position", "recent_team",
                      "games", "total_ppr", "proj_points"]].copy()
projections.rename(columns={"recent_team": "team"}, inplace=True)
projections = projections.sort_values("proj_points", ascending=False).reset_index(drop=True)

print(f"Veteran projections: {len(projections)} players")
print(f"\nTop 15 by projected points:")
display(projections.head(15).to_string(index=False))

Veteran projections: 506 players

Top 15 by projected points:


' player_id player_name position team  games  total_ppr  proj_points\n00-0034796   L.Jackson       QB  BAL     17     428.38   428.380000\n00-0036900     J.Chase       WR  CIN     17     403.00   403.000000\n00-0034857     J.Allen       QB  BUF     16     377.04   400.605000\n00-0036442    J.Burrow       QB  CIN     17     372.82   372.820000\n00-0034844   S.Barkley       RB  PHI     16     349.30   371.131250\n00-0034855  B.Mayfield       QB   TB     17     363.80   363.800000\n00-0039139     J.Gibbs       RB  DET     17     362.90   362.900000\n00-0036389     J.Hurts       QB  PHI     15     315.12   357.136000\n00-0039910   J.Daniels       QB  WAS     17     349.82   349.820000\n00-0038542  B.Robinson       RB  ATL     17     339.70   339.700000\n00-0032764     D.Henry       RB  BAL     17     336.40   336.400000\n00-0033921    C.Godwin       WR   TB      7     137.80   334.657143\n00-0033106      J.Goff       QB  DET     17     322.46   322.460000\n00-0033906    A.Kamara       RB  

## Cell 4 — Rookie & Missing Player Imputation via Sleeper

The nflreadpy historical data has no stats for **rookies** (years_exp == 0)
and some veteran free agents.  We use the cached Sleeper player catalog
to identify high-profile players missing from our veteran set and impute
baseline projections using a standard positional expectation curve.

In [4]:
# -- Load Sleeper catalog -------------------------------------------
assert os.path.exists(SLEEPER_CACHE), f"Missing: {SLEEPER_CACHE}"
with open(SLEEPER_CACHE) as f:
    sleeper_raw = json.load(f)

sleeper_df = pd.DataFrame.from_dict(sleeper_raw, orient="index")
sleeper_df = sleeper_df.reset_index(drop=True)
print(f"Sleeper catalog: {len(sleeper_df)} players")

# Align column names with nflreadpy convention
if "full_name" in sleeper_df.columns and "player_name" not in sleeper_df.columns:
    sleeper_df.rename(columns={"full_name": "player_name"}, inplace=True)

# Filter to active skill-position players with a reasonable search rank
sleeper_active = sleeper_df[
    (sleeper_df["status"] == "Active")
    & (sleeper_df["position"].isin(SKILL_POSITIONS))
    & (sleeper_df["search_rank"] <= 400)
].copy()
print(f"Active skill players (rank <= 400): {len(sleeper_active)}")

# -- Identify players missing from our veteran set ------------------
existing_ids = set(projections["player_id"].astype(str))
sleeper_active["player_id_str"] = sleeper_active["player_id"].astype(str)

missing = sleeper_active[
    ~sleeper_active["player_id_str"].isin(existing_ids)
].copy()
print(f"\nMissing from veteran set: {len(missing)}")
print(f"  Rookies (years_exp=0):  {len(missing[missing['years_exp'] == 0])}")
print(f"  Vets  (years_exp>0):    {len(missing[missing['years_exp'] > 0])}")

Sleeper catalog: 12223 players
Active skill players (rank <= 400): 530

Missing from veteran set: 530
  Rookies (years_exp=0):  55
  Vets  (years_exp>0):    475


In [5]:
def impute_projection(row) -> float:
    """Return an imputed 17-game Full-PPR projection based on position
    and Sleeper search_rank."""
    rank = row["search_rank"]
    pos  = row["position"]

    if pos == "QB":
        if rank <= 5:    return 360.0
        if rank <= 10:   return 320.0
        if rank <= 15:   return 280.0
        if rank <= 20:   return 260.0
        if rank <= 30:   return 230.0
        return 200.0

    if pos == "RB":
        if rank <= 5:    return 290.0
        if rank <= 10:   return 265.0
        if rank <= 20:   return 235.0
        if rank <= 30:   return 210.0
        if rank <= 40:   return 185.0
        if rank <= 60:   return 160.0
        return 140.0

    if pos == "WR":
        if rank <= 5:    return 310.0
        if rank <= 10:   return 285.0
        if rank <= 20:   return 250.0
        if rank <= 30:   return 220.0
        if rank <= 40:   return 195.0
        if rank <= 60:   return 170.0
        return 145.0

    if pos == "TE":
        if rank <= 5:    return 230.0
        if rank <= 10:   return 195.0
        if rank <= 15:   return 170.0
        if rank <= 20:   return 150.0
        if rank <= 30:   return 130.0
        return 110.0

    return 0.0


missing["proj_points"] = missing.apply(impute_projection, axis=1)
missing["games"] = 0
missing["total_ppr"] = np.nan

print("Imputation sample (top 15 missing players by projection):")
imputed_sample = missing.nlargest(15, "proj_points")[
    ["player_name", "position", "team", "search_rank", "years_exp", "proj_points"]
]
print(imputed_sample.to_string(index=False))

Imputation sample (top 15 missing players by projection):
        player_name position team  search_rank  years_exp  proj_points
         Josh Allen       QB  BUF          4.0        8.0        360.0
         Drake Maye       QB   NE          8.0        2.0        320.0
      Ja'Marr Chase       WR  CIN          4.0        5.0        310.0
         Puka Nacua       WR  LAR          5.0        3.0        310.0
         James Cook       RB  BUF          5.0        4.0        290.0
Christian McCaffrey       RB   SF          5.0        9.0        290.0
     Bijan Robinson       RB  ATL          1.0        3.0        290.0
       Jahmyr Gibbs       RB  DET          1.0        3.0        290.0
    Jonathan Taylor       RB  IND          4.0        6.0        290.0
        CeeDee Lamb       WR  DAL         10.0        6.0        285.0
 Jaxon Smith-Njigba       WR  SEA          6.0        3.0        285.0
  Amon-Ra St. Brown       WR  DET          8.0        5.0        285.0
      Lamar Jackson

## Cell 5 — Export & Downstream Alignment

Combine veterans and imputed players, deduplicate, and export to
`data/projections.csv` and overwrite `data/projections_template.csv`.

In [6]:
# -- Combine veterans + imputed rookies -----------------------------
veterans = projections[["player_name", "position", "team", "proj_points",
                         "player_id", "games"]].copy()
veterans["source"] = "nflreadpy"

imputed = missing[["player_name", "position", "team", "proj_points",
                    "player_id", "games"]].copy()
imputed["source"] = "sleeper_imputed"

combined = pd.concat([veterans, imputed], ignore_index=True)
print(f"Combined (pre-dedup): {len(combined)} players")

combined = combined.sort_values("proj_points", ascending=False)
combined = combined.drop_duplicates(subset=["player_name", "position"], keep="first")
print(f"After dedup:         {len(combined)} players")

combined = combined.sort_values("proj_points", ascending=False).reset_index(drop=True)
combined.index += 1
combined.index.name = "rank"

output = combined[["player_name", "position", "team", "proj_points"]].copy()
output["proj_points"] = output["proj_points"].round(1)

print(f"\nFinal projection set: {len(output)} players")

print("\nPosition distribution:")
pos_counts = output["position"].value_counts().sort_index()
for pos, cnt in pos_counts.items():
    print(f"  {pos}: {cnt}")

print("\nTop 10 per position:")
for pos in ["QB", "RB", "WR", "TE"]:
    subset = output[output["position"] == pos].head(10)
    print(f"\n  {pos}:")
    for _, r in subset.iterrows():
        name = r['player_name']
        team = r['team']
        pts  = r['proj_points']
        print(f"    {name:<24s}  {team:<4s}  {pts:6.1f}")

Combined (pre-dedup): 1036 players
After dedup:         1032 players

Final projection set: 1032 players

Position distribution:
  QB: 147
  RB: 260
  TE: 201
  WR: 424

Top 10 per position:

  QB:
    L.Jackson                 BAL    428.4
    J.Allen                   BUF    400.6
    J.Burrow                  CIN    372.8
    B.Mayfield                TB     363.8
    Josh Allen                BUF    360.0
    J.Hurts                   PHI    357.1
    J.Daniels                 WAS    349.8
    J.Goff                    DET    322.5
    Drake Maye                NE     320.0
    B.Nix                     DEN    317.2

  RB:
    S.Barkley                 PHI    371.1
    J.Gibbs                   DET    362.9
    B.Robinson                ATL    339.7
    D.Henry                   BAL    336.4
    A.Kamara                  NO     322.2
    D.Achane                  MIA    299.9
    J.Taylor                  IND    297.1
    J.Jacobs                  GB     293.1
    J.Mixon          

In [7]:
# -- Export to CSV ---------------------------------------------------
output.to_csv(PROJECTIONS_CSV, index=False)
print(f"Saved: {PROJECTIONS_CSV}  ({len(output)} rows)")

output.to_csv(TEMPLATE_CSV, index=False)
print(f"Saved: {TEMPLATE_CSV}  ({len(output)} rows)")

print(f"\nProjection range: {output['proj_points'].min():.1f} – {output['proj_points'].max():.1f} pts")
print(f"Mean:  {output['proj_points'].mean():.1f} pts")
print(f"Median: {output['proj_points'].median():.1f} pts")

Saved: /home/hadev/Projects/Lab/fantasy-footbal-analytics/data/projections.csv  (1032 rows)
Saved: /home/hadev/Projects/Lab/fantasy-footbal-analytics/data/projections_template.csv  (1032 rows)

Projection range: -5.9 – 428.4 pts
Mean:  134.4 pts
Median: 140.0 pts


## Cell 6 — End-to-End Validation

Verify the exported file is valid: non-empty, 250+ rows,
zero nulls in `proj_points`, and all 4 skill positions present.

In [8]:
print("=" * 60)
print("  END-TO-END VALIDATION")
print("=" * 60)

df = pd.read_csv(PROJECTIONS_CSV)
errors = []

if len(df) == 0:
    errors.append("FAIL: File is empty")
else:
    print(f"\n[PASS] Non-empty: {len(df)} rows")

if len(df) < 250:
    errors.append(f"FAIL: Only {len(df)} rows (need 250+)")
    print(f"[WARN] Row count {len(df)} < 250 target")
else:
    print(f"[PASS] Row count: {len(df)} (>= 250)")

null_count = df["proj_points"].isna().sum()
if null_count > 0:
    errors.append(f"FAIL: {null_count} null values in proj_points")
else:
    print(f"[PASS] Zero null values in proj_points")

positions_found = set(df["position"].unique())
expected = {"QB", "RB", "WR", "TE"}
missing_pos = expected - positions_found
if missing_pos:
    errors.append(f"FAIL: Missing positions: {missing_pos}")
else:
    print(f"[PASS] All 4 skill positions present: {sorted(positions_found)}")

expected_cols = {"player_name", "position", "team", "proj_points"}
actual_cols = set(df.columns)
if not expected_cols.issubset(actual_cols):
    errors.append(f"FAIL: Missing columns: {expected_cols - actual_cols}")
else:
    print(f"[PASS] Schema matches: {sorted(expected_cols)}")

if (df["proj_points"] <= 0).any():
    print(f"[WARN] {(df['proj_points'] <= 0).sum()} projections <= 0")
else:
    print(f"[PASS] All projections positive")

print("\n" + "=" * 60)
if errors:
    print("  RESULT: ISSUES FOUND")
    for e in errors:
        print(f"  • {e}")
else:
    print("  RESULT: ALL CHECKS PASSED")
print("=" * 60)

  END-TO-END VALIDATION

[PASS] Non-empty: 1032 rows
[PASS] Row count: 1032 (>= 250)
[PASS] Zero null values in proj_points
[PASS] All 4 skill positions present: ['QB', 'RB', 'TE', 'WR']
[PASS] Schema matches: ['player_name', 'position', 'proj_points', 'team']
[WARN] 23 projections <= 0

  RESULT: ALL CHECKS PASSED
